# Outil de gestion de bibliothèque

On commence ici un outil de gestion des emprunts d'une bibliothèque, avec l'ORM de SQLAlchemy sur une base SQLite : modéliser les livres, les membres et leurs emprunts, importer des données depuis des fichiers CSV, interroger la base, exporter les emprunts en CSV, puis les analyser avec pandas. Il suppose la partie ORM du TP « Prise en main de SQLAlchemy ».

Les étapes s'enchaînent : chaque solution définit ce que la suite utilise. Si vous êtes bloqué, exécutez la solution avant de passer à la suite.

## Mise en place

Les trois cellules suivantes écrivent les données de la bibliothèque, une table par fichier CSV : les livres (`livres.csv`), les membres (`membres.csv`), et les emprunts datés (`emprunts.csv`), dont certains sont rendus (ils ont une date de `retour`). La suivante crée le moteur, sur le fichier `bibliotheque.db`, et la classe `Base` des modèles.

In [ ]:
%%writefile livres.csv
titre,auteur
Vingt Mille Lieues sous les mers,Jules Verne
Voyage au centre de la Terre,Jules Verne
De la Terre à la Lune,Jules Verne
Le Tour du monde en quatre-vingts jours,Jules Verne
Michel Strogoff,Jules Verne
The Hound of the Baskervilles,Arthur Conan Doyle
"Arsène Lupin, gentleman-cambrioleur",Maurice Leblanc

In [ ]:
%%writefile membres.csv
nom
John Doe
Jane Doe
Sherlock Holmes
Arsène Lupin

In [ ]:
%%writefile emprunts.csv
livre,membre,date,retour
Voyage au centre de la Terre,Jane Doe,2026-01-03,2026-01-24
Le Tour du monde en quatre-vingts jours,Sherlock Holmes,2026-01-05,2026-01-19
Michel Strogoff,Sherlock Holmes,2026-01-07,
"Arsène Lupin, gentleman-cambrioleur",Jane Doe,2026-01-20,2026-02-01
The Hound of the Baskervilles,Sherlock Holmes,2026-02-02,
Voyage au centre de la Terre,John Doe,2026-02-10,
Le Tour du monde en quatre-vingts jours,Sherlock Holmes,2026-02-20,2026-03-06

In [ ]:
from sqlalchemy import create_engine, select
from sqlalchemy.orm import DeclarativeBase, Session

engine = create_engine("sqlite:///bibliotheque.db")


class Base(DeclarativeBase):
  pass

## 1. Modéliser livres, membres et emprunts

Un membre emprunte plusieurs livres, et un livre est emprunté par plusieurs membres au fil du temps : c'est une relation plusieurs-à-plusieurs. Chaque emprunt porte en plus ses propres informations, sa date et sa date de retour : il devient une classe à part entière, une *classe d'association*. Décrivez, sur la `Base` fournie :

- `Livre` (table `livres`) : `id` ; `titre`, unique ; `auteur` ; `emprunts`, la liste de ses emprunts ;
- `Membre` (table `membres`) : `id` ; `nom`, unique ; `emprunts` ;
- `Emprunt` (table `emprunts`) : `id` ; `id_livre` et `id_membre`, clefs étrangères ; `date_emprunt`, une date ; `date_retour`, une date facultative (un emprunt en cours n'en a pas) ; `livre` et `membre`, l'objet de chaque côté.

Supprimez les tables existantes, puis créez celles-ci. Pourquoi un emprunt a-t-il son propre `id`, plutôt qu'une clef primaire formée de `id_livre` et `id_membre` ?

**Documentation utile**

- [Classe d'association](https://docs.sqlalchemy.org/en/21/orm/basic_relationships.html#association-object)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Une date s'annote `Mapped[date]` (`from datetime import date`), une colonne facultative `Mapped[date | None]`, et `mapped_column(unique=True)` interdit les doublons. Il y a deux relations, chacune avec ses deux côtés, qui se nomment l'un l'autre par `back_populates` : `Livre.emprunts` et `Emprunt.livre`, `Membre.emprunts` et `Emprunt.membre`.

```python
from datetime import date

from sqlalchemy import ForeignKey
from sqlalchemy.orm import Mapped, mapped_column, relationship


class Livre(Base):
  __tablename__ = "livres"
  id: Mapped[int] = mapped_column(primary_key=True)
  titre: Mapped[str] = mapped_column(unique=True)
  auteur: Mapped[str]
  emprunts: Mapped[list["Emprunt"]] = relationship(back_populates="livre")


class Membre(Base):
  __tablename__ = "membres"
  ...


class Emprunt(Base):
  __tablename__ = "emprunts"
  id: Mapped[int] = mapped_column(primary_key=True)
  id_livre: Mapped[int] = mapped_column(ForeignKey(...))
  ...
  livre: Mapped[Livre] = relationship(back_populates=...)
  ...


Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)
```

### Solution

In [ ]:
from datetime import date

from sqlalchemy import ForeignKey
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship


class Base(DeclarativeBase):
  pass


class Livre(Base):
  __tablename__ = "livres"
  id: Mapped[int] = mapped_column(primary_key=True)
  titre: Mapped[str] = mapped_column(unique=True)
  auteur: Mapped[str]
  emprunts: Mapped[list["Emprunt"]] = relationship(back_populates="livre")


class Membre(Base):
  __tablename__ = "membres"
  id: Mapped[int] = mapped_column(primary_key=True)
  nom: Mapped[str] = mapped_column(unique=True)
  emprunts: Mapped[list["Emprunt"]] = relationship(back_populates="membre")


class Emprunt(Base):
  __tablename__ = "emprunts"
  id: Mapped[int] = mapped_column(primary_key=True)
  id_livre: Mapped[int] = mapped_column(ForeignKey("livres.id"))
  id_membre: Mapped[int] = mapped_column(ForeignKey("membres.id"))
  date_emprunt: Mapped[date]
  date_retour: Mapped[date | None]
  livre: Mapped[Livre] = relationship(back_populates="emprunts")
  membre: Mapped[Membre] = relationship(back_populates="emprunts")


Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# La solution part d'une Base neuve : déclarer une seconde fois une classe Livre
# sur la même Base (si vous l'avez déjà fait plus haut) lèverait une erreur.
#
# Un même membre peut emprunter plusieurs fois le même livre (dans les données,
# Sherlock Holmes et Le Tour du monde en quatre-vingts jours) : le couple
# (id_livre, id_membre) n'identifie pas un emprunt, et ne peut donc pas servir de
# clef primaire. D'où un id propre à chaque emprunt.

## 2. Importer les fichiers CSV

Complétez la fonction `importer_csv(dossier, session)`, qui lit les fichiers `livres.csv`, `membres.csv` et `emprunts.csv` du dossier `dossier`, et ajoute à la session un `Livre` par livre, un `Membre` par membre et un `Emprunt` par emprunt. Les emprunts désignent livres et membres par leur titre et leur nom : gardez les objets créés dans des dictionnaires pour les retrouver. Appelez-la ensuite sur le dossier courant (`Path(".")`) dans une session, validez, puis affichez le nombre de livres, de membres et d'emprunts de la base.

**Documentation utile**

- [`csv.DictReader`](https://docs.python.org/fr/3/library/csv.html#csv.DictReader)
- [`date.fromisoformat`](https://docs.python.org/fr/3/library/datetime.html#datetime.date.fromisoformat)

In [ ]:
from pathlib import Path


def importer_csv(dossier: Path, session: Session) -> None:
  """Ajoute à la session les livres, membres et emprunts des CSV de dossier."""
  ...

In [ ]:
# Votre code ici

### Conseils supplémentaires

`csv.DictReader(f)` lit un fichier CSV ligne à ligne, chaque ligne en dictionnaire dont les clefs sont les noms de colonnes de l'en-tête ; ouvrez le fichier avec `newline=""`, comme le demande le module `csv`. Le titre qui contient une virgule est entre guillemets dans le fichier : `csv` les retire. Toutes les valeurs sont des chaînes : `date.fromisoformat` convertit les dates, et `retour` est une chaîne vide pour un emprunt en cours. Il suffit d'ajouter à la session les livres, les membres et les emprunts : les relations font le lien. Vous devez obtenir 7 livres, 4 membres et 7 emprunts.

```python
import csv


def lire_csv(chemin: Path) -> list[dict[str, str]]:
  with open(chemin, newline="", encoding="utf-8") as f:
    return list(csv.DictReader(f))


def importer_csv(dossier: Path, session: Session) -> None:
  livres = {
    ligne["titre"]: Livre(titre=ligne["titre"], auteur=ligne["auteur"])
    for ligne in lire_csv(dossier / "livres.csv")
  }
  membres = {
    ligne["nom"]: Membre(nom=ligne["nom"])
    for ligne in lire_csv(dossier / "membres.csv")
  }
  session.add_all(livres.values())
  session.add_all(membres.values())
  for ligne in lire_csv(dossier / "emprunts.csv"):
    retour = ligne["retour"]
    session.add(
      Emprunt(
        livre=...,
        membre=...,
        date_emprunt=...,
        date_retour=...,
      )
    )


with Session(engine) as session:
  importer_csv(Path("."), session)
  session.commit()
  for modele in (Livre, Membre, Emprunt):
    print(modele.__name__, len(session.scalars(select(modele)).all()))
```

### Solution

In [ ]:
import csv


def lire_csv(chemin: Path) -> list[dict[str, str]]:
  with open(chemin, newline="", encoding="utf-8") as f:
    return list(csv.DictReader(f))


def importer_csv(dossier: Path, session: Session) -> None:
  livres = {
    ligne["titre"]: Livre(titre=ligne["titre"], auteur=ligne["auteur"])
    for ligne in lire_csv(dossier / "livres.csv")
  }
  membres = {
    ligne["nom"]: Membre(nom=ligne["nom"])
    for ligne in lire_csv(dossier / "membres.csv")
  }
  session.add_all(livres.values())
  session.add_all(membres.values())
  for ligne in lire_csv(dossier / "emprunts.csv"):
    retour = ligne["retour"]
    session.add(
      Emprunt(
        livre=livres[ligne["livre"]],
        membre=membres[ligne["membre"]],
        date_emprunt=date.fromisoformat(ligne["date"]),
        date_retour=date.fromisoformat(retour) if retour else None,
      )
    )


with Session(engine) as session:
  importer_csv(Path("."), session)
  session.commit()
  for modele in (Livre, Membre, Emprunt):
    print(modele.__name__, len(session.scalars(select(modele)).all()))

## 3. Interroger la base

Chaque requête de cette partie s'écrit avec `select`, et s'exécute dans une session avec `session.scalars(...)`.

### Les livres empruntés par Sherlock Holmes

Affichez le titre des livres que Sherlock Holmes a empruntés, avec la date de chaque emprunt, triés par date.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Partez de `select(Emprunt)`, joignez `Emprunt.membre`, filtrez sur `Membre.nom` et triez par `Emprunt.date_emprunt` avec `order_by`. Chaque emprunt donne son livre par `emprunt.livre`. Vous devez obtenir quatre emprunts, dont deux du même livre, du 5 janvier au 20 février 2026.

```python
with Session(engine) as session:
  requete = select(Emprunt).join(...).where(...).order_by(...)
  for emprunt in session.scalars(requete):
    print(emprunt.date_emprunt, emprunt.livre.titre)
```

#### Solution

In [ ]:
with Session(engine) as session:
  requete = (
    select(Emprunt)
    .join(Emprunt.membre)
    .where(Membre.nom == "Sherlock Holmes")
    .order_by(Emprunt.date_emprunt)
  )
  for emprunt in session.scalars(requete):
    print(emprunt.date_emprunt, emprunt.livre.titre)

### Les emprunts en cours

Affichez les emprunts en cours, ceux qui n'ont pas de date de retour, par date d'emprunt : la date, le titre du livre et le nom du membre.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

En SQL, l'absence de valeur se teste avec `IS NULL`, que SQLAlchemy écrit `Emprunt.date_retour.is_(None)`. Vous devez obtenir 3 emprunts.

```python
with Session(engine) as session:
  requete = select(Emprunt).where(...).order_by(Emprunt.date_emprunt)
  for emprunt in session.scalars(requete):
    print(...)
```

#### Solution

In [ ]:
with Session(engine) as session:
  en_cours = Emprunt.date_retour.is_(None)
  requete = select(Emprunt).where(en_cours).order_by(Emprunt.date_emprunt)
  for emprunt in session.scalars(requete):
    print(emprunt.date_emprunt, emprunt.livre.titre, emprunt.membre.nom)

### Les livres disponibles

Affichez, par ordre alphabétique, le titre des livres disponibles : ceux qui n'ont aucun emprunt en cours.

**Documentation utile**

- [`any`](https://docs.sqlalchemy.org/en/21/orm/internals.html#sqlalchemy.orm.RelationshipProperty.Comparator.any)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`Livre.emprunts.any(condition)` est vrai pour un livre qui a au moins un emprunt vérifiant la condition (en SQL, un `EXISTS` avec un `WHERE`), et `~` en prend la négation. La condition « en cours » est celle de l'exercice précédent. Attention : un livre rendu peut avoir été emprunté de nouveau. Vous devez obtenir 4 livres.

```python
with Session(engine) as session:
  en_cours = Emprunt.date_retour.is_(None)
  requete = select(Livre).where(...).order_by(Livre.titre)
  for livre in session.scalars(requete):
    print(livre.titre)
```

#### Solution

In [ ]:
with Session(engine) as session:
  en_cours = Emprunt.date_retour.is_(None)
  requete = select(Livre).where(~Livre.emprunts.any(en_cours)).order_by(Livre.titre)
  for livre in session.scalars(requete):
    print(livre.titre)

# Voyage au centre de la Terre n'y est pas : rendu par Jane Doe, il a été emprunté
# de nouveau, par John Doe. Un livre est disponible s'il n'a aucun emprunt en
# cours, et non s'il a un emprunt rendu.

## 4. Exporter les emprunts en CSV

Complétez la fonction `exporter_csv(chemin, session)`, qui écrit tous les emprunts, par date d'emprunt, dans le fichier CSV `chemin` : une ligne par emprunt, avec les colonnes `livre`, `membre`, `date_emprunt` et `date_retour` (vide pour un emprunt en cours). Appelez-la pour écrire `export.csv`, puis affichez le contenu du fichier.

**Documentation utile**

- [`csv.DictWriter`](https://docs.python.org/fr/3/library/csv.html#csv.DictWriter)
- [`date.isoformat`](https://docs.python.org/fr/3/library/datetime.html#datetime.date.isoformat)

In [ ]:
def exporter_csv(chemin: str, session: Session) -> None:
  """Écrit les emprunts de la base dans le fichier CSV chemin."""
  ...

In [ ]:
# Votre code ici

### Conseils supplémentaires

`csv.DictWriter(f, fieldnames=[...])` écrit des dictionnaires : `writeheader()` écrit l'en-tête, puis `writerow(...)` chaque ligne. Ouvrez le fichier avec `newline=""`, comme le demande le module `csv`. `isoformat()` écrit une date au format `2026-01-03`. Le fichier doit avoir 8 lignes : l'en-tête et 7 emprunts.

```python
import csv


def exporter_csv(chemin: str, session: Session) -> None:
  with open(chemin, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
      f, fieldnames=["livre", "membre", "date_emprunt", "date_retour"]
    )
    writer.writeheader()
    for emprunt in session.scalars(select(Emprunt).order_by(Emprunt.date_emprunt)):
      writer.writerow(
        {
          "livre": emprunt.livre.titre,
          "membre": ...,
          "date_emprunt": ...,
          "date_retour": ...,
        }
      )


with Session(engine) as session:
  exporter_csv("export.csv", session)

with open("export.csv", encoding="utf-8") as f:
  print(f.read())
```

### Solution

In [ ]:
import csv


def exporter_csv(chemin: str, session: Session) -> None:
  with open(chemin, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
      f, fieldnames=["livre", "membre", "date_emprunt", "date_retour"]
    )
    writer.writeheader()
    for emprunt in session.scalars(select(Emprunt).order_by(Emprunt.date_emprunt)):
      retour = emprunt.date_retour
      writer.writerow(
        {
          "livre": emprunt.livre.titre,
          "membre": emprunt.membre.nom,
          "date_emprunt": emprunt.date_emprunt.isoformat(),
          "date_retour": retour.isoformat() if retour else "",
        }
      )


with Session(engine) as session:
  exporter_csv("export.csv", session)

with open("export.csv", encoding="utf-8") as f:
  print(f.read())

## 5. Analyser les emprunts avec pandas

Avec `pd.read_sql`, chargez dans un `DataFrame` `emprunts` le résultat d'une requête `select` qui donne, pour chaque emprunt, le nom du membre (colonne `membre`), le titre du livre (colonne `livre`) et la date d'emprunt. Affichez ensuite le nombre d'emprunts de chaque membre, du plus grand au plus petit.

**Documentation utile**

- [`ColumnElement.label`](https://docs.sqlalchemy.org/en/21/core/sqlelement.html#sqlalchemy.sql.expression.ColumnElement.label)
- [`Series.value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`pd.read_sql(requete, engine)` exécute la requête et renvoie un `DataFrame`, une colonne par élément du `select`, et `.label("membre")` nomme une colonne. Partez de l'emprunt, `select_from(Emprunt)`, pour joindre ses deux relations. `value_counts()` compte les valeurs d'une colonne, de la plus fréquente à la moins fréquente. Vous devez obtenir Sherlock Holmes 4, Jane Doe 2 et John Doe 1.

```python
import pandas as pd

requete = (
  select(Membre.nom.label("membre"), ..., Emprunt.date_emprunt)
  .select_from(Emprunt)
  .join(...)
  .join(...)
)
emprunts = pd.read_sql(requete, engine)
print(...)
```

### Solution

In [ ]:
import pandas as pd

requete = (
  select(Membre.nom.label("membre"), Livre.titre.label("livre"), Emprunt.date_emprunt)
  .select_from(Emprunt)
  .join(Emprunt.membre)
  .join(Emprunt.livre)
)
emprunts = pd.read_sql(requete, engine)
print(emprunts["membre"].value_counts())